# 🧠 Aula 06 — Sistemas Operacionais Linux e GPU

**Objetivo:** utilizar comandos e estruturas do Linux para gerenciar sistemas com GPUs,
preparando e automatizando o ambiente para execução de cargas de trabalho de IA.

**Roteiro deste notebook:**
1. Verificação do ambiente e exploração do sistema de arquivos.
2. Teoria: a estrutura de diretórios relevante para GPUs.
3. Demo: identificar GPUs e drivers (`lspci`, `nvidia-smi`/`rocm-smi`).
4. Atividade: ler temperatura/utilização e disparar alerta.
5. Automação: `cron` (periódico) e `systemd` (contínuo).
6. Discussão e síntese.

> 💡 **Sem GPU?** O notebook detecta e mostra um exemplo simulado — a aula roda do
> começo ao fim. Muitos comandos usam `!` (shell), então só executam no **Colab/Linux**.

## 1. Verificação do Ambiente

O Linux expõe hardware e kernel como **pseudo-arquivos** (não ocupam disco). Vamos explorar
os diretórios-chave.

In [ ]:
# @title 🐧 Explorar o sistema de arquivos (Colab/Linux)
# ============================================================================
# OBJETIVO: conhecer a raiz do sistema e os diretórios que importam para GPU.
# Obs.: linhas com '!' rodam comandos do shell (só funcionam no Colab/Linux).
# ============================================================================
!ls /
print("\n--- dispositivos NVIDIA (se houver) ---")
!ls /dev/nvidia* 2>/dev/null || echo "sem /dev/nvidia*"
print("\n--- dispositivos gráficos (DRM) ---")
!ls /dev/dri/ 2>/dev/null || echo "sem /dev/dri"
print("\n--- GPUs via sysfs ---")
!ls /sys/class/drm/ 2>/dev/null || echo "sem /sys/class/drm"

## 2. Teoria: estrutura de diretórios

| Caminho | O que contém |
| :--- | :--- |
| `/dev/nvidia*` | Arquivos de dispositivo das GPUs NVIDIA (`nvidia0`, `nvidiactl`) |
| `/proc/driver/nvidia/` | Info do driver em tempo real: versão, GPUs, clients |
| `/sys/class/drm/` | Interface sysfs para GPUs via DRM (vendor, classe) |
| `/usr/lib/x86_64-linux-gnu/` | Bibliotecas do sistema (`libcuda.so`, `libnvidia*.so`) |
| `/etc/modprobe.d/` | Configuração de módulos do kernel (blacklist, opções) |
| `/var/log/` | Logs: `syslog`, `kern.log`, `nvidia-installer.log` |

**Pseudo-arquivos:** `/proc` e `/sys` são uma janela viva para o kernel. `htop` e `nvtop`
leem exatamente daí.

In [ ]:
# @title 📊 Ler CPU, memória e uptime dos pseudo-arquivos
# ============================================================================
# OBJETIVO: ler /proc/cpuinfo, /proc/meminfo e /proc/uptime como arquivos de
# texto comuns — a base do monitoramento de servidores de IA.
# ============================================================================
import os

if os.path.isdir("/proc"):
    with open("/proc/cpuinfo") as f:
        for linha in f:
            if "model name" in linha:
                print(f"CPU: {linha.split(':', 1)[1].strip()}")
                break
    with open("/proc/meminfo") as f:
        for linha in f:
            if linha.startswith(("MemTotal", "MemAvailable")):
                chave, valor = linha.split(":", 1)
                print(f"{chave}: {int(valor.split()[0]) / 1024:,.0f} MB")
    with open("/proc/uptime") as f:
        print(f"Ligado há: {float(f.read().split()[0]) / 3600:.1f} horas")
else:
    print("Este ambiente não tem /proc (não é Linux).")
    print("No Windows, use o script scripts/monitoramento_linux.py (via psutil).")

## 3. Demo: identificando GPUs e drivers

O **primeiro passo** num servidor novo: ver se a placa existe (`lspci`) e se o driver
responde (`nvidia-smi` / `rocm-smi`). Driver ausente é a causa nº 1 de falhas em IA.

In [ ]:
# @title 🔎 Identificar GPUs e verificar drivers
# ============================================================================
# OBJETIVO: localizar a GPU no barramento PCI e checar o driver.
# ============================================================================
import shutil, subprocess

print("[lspci] dispositivos gráficos:")
if shutil.which("lspci"):
    saida = subprocess.run(["lspci"], capture_output=True, text=True).stdout
    for linha in saida.splitlines():
        if any(t in linha.lower() for t in ("vga", "3d", "display", "nvidia", "amd", "radeon")):
            print("  " + linha)
else:
    print("  lspci não disponível.")

print("\n[nvidia-smi]")
if shutil.which("nvidia-smi"):
    saida = subprocess.run(
        ["nvidia-smi", "--query-gpu=index,name,driver_version,memory.total",
         "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
    print("  " + saida)
else:
    print("  nvidia-smi não encontrado (sem GPU NVIDIA ou driver ausente).")

print("\n[rocm-smi] (AMD no Linux)")
print("  " + ("disponível" if shutil.which("rocm-smi") else "não encontrado"))

## 4. Atividade: status e alerta de temperatura

Lemos temperatura, utilização e memória de cada GPU e disparamos um **alerta** acima de um
limite (80°C). É a lógica do script `gpu_status.sh`, em Python, para rodar aqui também.

In [ ]:
# @title 🌡️ Status das GPUs + alerta de temperatura
# ============================================================================
# OBJETIVO: mostrar o status de cada GPU e alertar se passar do limite.
# Usa nvidia-smi se existir; senão, exibe um exemplo simulado.
# ============================================================================
import shutil, subprocess

LIMITE_TEMP = 80   # °C

if shutil.which("nvidia-smi"):
    saida = subprocess.run(
        ["nvidia-smi",
         "--query-gpu=index,name,temperature.gpu,utilization.gpu,memory.used,memory.total,power.draw",
         "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout.strip()
    linhas = saida.splitlines()
else:
    print("NVIDIA-smi indisponível — exemplo simulado.\n")
    # mesmo formato do nvidia-smi: idx,nome,temp,util,usada,total,watts
    linhas = ["0, Tesla T4, 72, 87, 11570, 15360, 48.2"]

for linha in linhas:
    idx, nome, temp, util, usada, total, watts = [p.strip() for p in linha.split(",")]
    print(f"GPU {idx}: {nome}")
    print(f"  Temperatura : {temp} °C")
    print(f"  Utilização  : {util} %")
    print(f"  Memória     : {usada} / {total} MB")
    print(f"  Consumo     : {watts} W")
    # Alerta: converte para número e compara com o limite.
    if int(float(temp)) > LIMITE_TEMP:
        print(f"  ⚠️  ALERTA: temperatura acima de {LIMITE_TEMP}°C!")
    print()

print("→ Em produção, este alerta dispara um e-mail/webhook.")
print("→ Agende a leitura com cron (a cada 5 min) ou como serviço systemd.")

## 5. Automação: cron e systemd

| | **cron** | **systemd** |
| :--- | :--- | :--- |
| Tipo | Periódico (agenda) | Contínuo (serviço) |
| Reinicia sozinho | Não | **Sim** |
| Inicia no boot | Não | **Sim** |
| Exemplo | Relatório 08:00 | Monitor de GPU |

A célula abaixo apenas **mostra** os arquivos de exemplo (cron e systemd). Não os instala —
no Colab não há `cron` nem `systemd` de verdade.

In [ ]:
# @title ⚙️ Exemplos de automação (cron e systemd)
# ============================================================================
# OBJETIVO: ver o formato de uma linha de cron e de um serviço systemd.
# (No Colab não há cron/systemd; aqui apenas imprimimos os exemplos.)
# ============================================================================
print("Linha de crontab — monitorar a GPU a cada 5 minutos:")
print("  */5 * * * * /home/usuario/scripts/gpu_status.sh >> /var/log/gpu_monitor.log 2>&1")
print()
print("Formato:  minuto  hora  dia  mes  dia_semana  comando")
print("          */5     *     *    *    *           comando")
print()
print("Unidade systemd — monitor contínuo (gpu-monitor.service):")
print("  [Unit]")
print("  Description=Monitor de GPU")
print("  [Service]")
print("  ExecStart=/usr/bin/python3 /home/usuario/scripts/monitoramento_linux.py")
print("  Restart=always")
print("  [Install]")
print("  WantedBy=multi-user.target")
print()
print("Ativar:  sudo systemctl enable --now gpu-monitor")

## 6. Discussão em Grupo

Em grupos de 3–4, no cenário do servidor da startup:

1. O treino rodava há 12h quando o SSH caiu. Como evitar? Quais ferramentas usar?
2. Dois cientistas querem usar as mesmas 4 GPUs. Como gerenciar acesso e recursos?
3. O script mostra 85°C numa GPU. Próximos passos? Como automatizar o alerta?
4. Por que `/proc` e `/sys` são "sistemas de arquivos virtuais"? O que isso muda?

> Atividade de pesquisa completa em `aulas/aula06/atividade.md`.

## 7. Exercícios (5)

Resolva os 5 exercícios abaixo **neste notebook** (Colab). Cada um reforça uma ideia da
aula. Não há resposta única: o valor está em **experimentar e explicar** o resultado.

---

**1) Pseudo-arquivos vs. arquivos reais.** No Colab, liste `/proc` e `/sys` com `!ls`.
Explique, com suas palavras, por que `/proc/cpuinfo` **não ocupa espaço em disco** e o que
isso representa.

**2) Você chegou no servidor novo.** Escreva a sequência de comandos para responder:
*(a)* qual placa de vídeo está instalada? *(b)* o driver está funcionando? *(c)* quanta
VRAM existe? Dica: `lspci`, `nvidia-smi`. Rode o que o Colab permitir.

**3) Alerta de temperatura.** Abaixo há uma célula-esqueleto. Complete a lógica para
**imprimir um alerta** quando a temperatura da GPU passar de **80°C** (use os dados
simulados se não houver GPU). Teste com 72°C e com 85°C.

**4) cron ou systemd?** Para cada caso, escolha a ferramenta e justifique em uma frase:
*(a)* enviar um relatório de uso da GPU todo dia às 08:00; *(b)* manter um monitor de GPU
sempre ativo, que reinicia sozinho se cair; *(c)* apagar logs antigos toda semana.

**5) O SSH caiu no meio do treino.** Descreva como você teria evitado a perda do job usando
**tmux** ou **screen**. Liste os 3 comandos principais (criar, desanexar, reanexar).


In [ ]:
# @title Exercício 3 — complete o alerta de temperatura
# ============================================================================
# OBJETIVO: imprimir ALERTA quando a temperatura passar de LIMITE_TEMP.
# Complete os TODOs. Rode com temperatura 72 e depois 85 para testar.
# ============================================================================
LIMITE_TEMP = 80

# Caso queira testar sem GPU, troque a linha abaixo por uma temperatura.
# Ex.: temperaturas = [72]   ou   temperaturas = [85]
temperaturas = [72, 85]

for temp in temperaturas:
    print(f"GPU a {temp}°C:", end=" ")
    if temp > LIMITE_TEMP:
        # TODO: imprima aqui o alerta (ex.: print com aviso)
        print("")
    else:
        print("dentro do limite")

# Dica: o resultado esperado é
#   GPU a 72°C: dentro do limite
#   GPU a 85°C: ALERTA: temperatura acima de 80°C!

## 8. Síntese e Tarefa de Casa

**O que levar:**
- **/dev, /proc, /sys:** pseudo-arquivos — janela para hardware e kernel em tempo real.
- **lspci / nvidia-smi:** identificar e verificar GPUs — primeiro passo num servidor novo.
- **Script Bash:** automatiza verificações repetitivas (status, alerta de temperatura).
- **screen/tmux/nohup:** mantêm treinos longos vivos após desconectar o SSH.
- **cron:** tarefas periódicas; **systemd:** serviços contínuos com reinício automático.

**Tarefa (opcional):** expanda o `gpu_status.sh` para incluir:
- envio de e-mail/webhook quando a temperatura &gt; 80°C;
- log estruturado em CSV com timestamp, temperatura e utilização;
- relatório da utilização média das últimas 24h;
- agendamento via cron a cada 5 minutos.

> 🔗 **Próxima aula:** *Introdução ao CUDA* — o ambiente está estável; agora vamos escrever
> kernels que rodam direto na VRAM.